# ansel-denoise — test training on Google Colab

Trains the CFA-domain denoising U-Net on the shard cache published on the
[`shards-v1` release](https://github.com/aurelienpierreeng/ansel-denoise/releases/tag/shards-v1),
with noise synthesized on the fly from the darktable/Ansel noise profiles.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Checkpoints go to your Google Drive (if you allow the mount below) and every
session trains `STEPS` **additional** steps on top of the latest checkpoint —
so after a free-tier disconnect, or to keep refining the same run, just run
all cells again. Shards are re-fetched each session (the release may have
grown since); the checkpoint continuity lives in Drive.

In [ ]:
# --- parameters -----------------------------------------------------------
STEPS = 10000       # steps to train THIS SESSION (adds to previous sessions); ~25-40 min on a T4
BATCH = 32
PATCH = 128
RUN_NAME = "colab-test"
USE_DRIVE = True    # persist checkpoints to Google Drive (asks for consent)

!nvidia-smi -L || echo 'NO GPU: switch the runtime type to T4 GPU'

In [ ]:
# --- code -----------------------------------------------------------------
# Anchored to an absolute path and pulled (not skipped) when already cloned,
# so a reused Colab runtime always runs the CURRENT code. Already-imported
# modules are purged before re-import for the same reason. No pip install:
# `pip install -e .` needs a kernel restart on Colab, and every dependency
# (numpy, torch) is preinstalled — src/ on sys.path is enough.
import os, sys
ROOT = '/content' if os.path.isdir('/content') else os.path.expanduser('~')
REPO = os.path.join(ROOT, 'ansel-denoise')
if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 https://github.com/aurelienpierreeng/ansel-denoise.git {REPO}
!git -C {REPO} log --oneline -1
%cd {REPO}
sys.path.insert(0, os.path.join(REPO, 'src'))
for m in [m for m in list(sys.modules) if m.startswith('ansel_denoise')]:
    del sys.modules[m]
import ansel_denoise
print('package importable:', ansel_denoise.__file__)

In [ ]:
# --- data: fetch the published shard cache (no auth, public release) ------
import json, tarfile, urllib.request
from pathlib import Path

SHARDS = Path('shards/rpu')
SHARDS.mkdir(parents=True, exist_ok=True)
api = 'https://api.github.com/repos/aurelienpierreeng/ansel-denoise/releases/tags/shards-v1'
with urllib.request.urlopen(api) as r:
    assets = json.load(r)['assets']
for a in assets:
    if not a['name'].startswith('shards-'):
        continue
    print(a['name'], f"{a['size'] / 1e6:.0f} MB")
    tmp, _ = urllib.request.urlretrieve(a['browser_download_url'])
    with tarfile.open(tmp) as tar:
        tar.extractall(SHARDS, filter='data')
    Path(tmp).unlink()
print(f"{len(list(SHARDS.glob('*.npz')))} shards ready")

In [ ]:
# --- checkpoints: Google Drive if allowed, else session-local -------------
from pathlib import Path

out_root = Path('runs')
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        out_root = Path('/content/drive/MyDrive/ansel-denoise-runs')
    except Exception as e:
        print(f'Drive not mounted ({e}); checkpoints stay in the session')
OUT = out_root / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

# Resume from the highest NUMBERED checkpoint (never ckpt-final.pt: it is a
# stable alias that could be staler than a mid-run numbered checkpoint).
# Each session trains STEPS *additional* steps on top of it.
ckpts = sorted(OUT.glob('ckpt-0*.pt'))
start = int(ckpts[-1].stem.split('-')[1]) if ckpts else 0
TARGET = start + STEPS
RESUME = ['--resume', str(ckpts[-1])] if ckpts else []
print(f'checkpoints -> {OUT} | training steps {start} -> {TARGET}')

In [ ]:
# --- train ----------------------------------------------------------------
# --schedule constant: additive sessions must NOT use the cosine schedule —
# it anneals toward the session target, so every later session would resume
# deep in the dying tail of the curve and train at lr ~ 0 (observed). Anneal
# deliberately with a final one-shot cosine run when the model is done.
# --patience 12: stop if 12 consecutive validations (6000 steps) bring no
# +0.05 dB val improvement, counted across sessions via the checkpoint.
from ansel_denoise.train import main

main(['--shards', 'shards/rpu', '--out', str(OUT),
      '--steps', str(TARGET), '--batch', str(BATCH), '--patch', str(PATCH),
      '--workers', '2', '--val-every', '500', '--ckpt-every', '500',
      '--schedule', 'constant', '--patience', '12', *RESUME])

In [ ]:
# --- export the weights for Ansel and download them -----------------------
from ansel_denoise.export import main as export_main
export_main([str(OUT / 'ckpt-final.pt'), '--out', str(OUT / 'final.anseldn')])
try:
    from google.colab import files
    files.download(str(OUT / 'final.anseldn'))
except Exception:
    print(f'weights at {OUT}/final.anseldn')